# parser_dev comparison notebook

This notebook is for old-vs-new comparison using the draft functions in `dev/edit_plan.py`.
It is development-only and is meant to make input/output differences easy to inspect.


In [1]:
from pathlib import Path
from pprint import pprint
import sys
import types
import os
import traceback
import numpy as np

try:
    import openbabel  # type: ignore
except ModuleNotFoundError:
    class _DummyErrorLog:
        def SetOutputLevel(self, *_args, **_kwargs):
            return None

    openbabel_stub = types.ModuleType('openbabel')
    openbabel_stub.pybel = types.SimpleNamespace()
    openbabel_stub.openbabel = types.SimpleNamespace(obErrorLog=_DummyErrorLog(), obError=0)
    sys.modules['openbabel'] = openbabel_stub

HERE = Path.cwd()
REPO = HERE.parent
TEST_DIR = REPO / 'test'
DEV_DIR = REPO / 'dev'
GENERATED_DIR = DEV_DIR / 'generated'
GENERATED_DIR.mkdir(exist_ok=True)

XYZ_HAA = TEST_DIR / 'molecules' / 'haa.xyz'
MOL_ETHANOL = TEST_DIR / 'molecules' / 'ethanol.mol'
MOL_ACETATE = TEST_DIR / 'molecules' / 'acetate.mol'

SMILES_CASES = {
    'haa_unmapped': 'O=CCO',
    'haa_mapped': '[C:0]([C:1]([O:7][H:11])([H:13])[H:14])(=[O:6])[H:12]',
    'aromatic_lowercase': 'Oc1ccoc1',
    'aromatic_mapped': '[c:0]1([H:15])[c:4]([O:9][H:13])[c:5]([H:20])[c:1]([H:16])[o:6]1',
    'naphthalene_lowercase': 'c1ccc2ccccc2c1',
    'stereo_case': 'F/C=C/F',
}


In [2]:
import edit_plan as edit_plan

from edit_plan import (
    smiles2adjmat_old,
    smiles2adjmat_new,
    mol_parse_old,
    mol_parse_new,
    xyz_from_smiles_old,
    xyz_from_smiles_new,
    yarpecule_read_structure_old,
    yarpecule_read_structure_new,
    yarpecule_get_smiles_old,
    yarpecule_get_smiles_new,
    yarpecule_export_geometry_old,
    yarpecule_export_geometry_new,
    mol_write_yp_new,
    xyz_write_new,
)

from yarp.yarpecule.graph.smiles import add_hydrogens, reorder_by_mappings, OctetError
from yarp.yarpecule.input_parsers import xyz_parse, xyz_q_parse, mol_parse, xyz_from_smiles
from yarp.yarpecule.graph.smiles import smiles2adjmat
from yarp.yarpecule.graph.adjacency import adjmat_to_adjlist, table_generator
from yarp.yarpecule.graph.fragment import return_rings
from yarp.yarpecule.atom_mapping import canon_order
from yarp.yarpecule.hashes import atom_hash
from yarp.yarpecule.yarpecule import yarpecule
from yarp.util.write_files import mol_write_yp, xyz_write, return_formals
from yarp.util.properties import el_valence, el_metals, el_expand_octet, el_to_an, el_n_expand_octet, el_mass

from rdkit.Chem import rdmolfiles, BondType, rdchem, Atom, MolFromSmiles, AddHs, AllChem
from rdkit import Chem

edit_plan.return_rings = return_rings
edit_plan.adjmat_to_adjlist = adjmat_to_adjlist
edit_plan.el_valence = el_valence
edit_plan.el_metals = el_metals
edit_plan.el_expand_octet = el_expand_octet
edit_plan.canon_order = canon_order
edit_plan.el_mass = el_mass


In [3]:
def xyz_from_smiles_new_bound(smiles, mode="yarp"):
    return xyz_from_smiles_new(
        smiles, mode, smiles2adjmat_new, BondType, MolFromSmiles, rdchem, Atom,
        el_to_an, el_n_expand_octet, el_expand_octet, OctetError, AddHs, AllChem, np, el_mass
    )

def mol_parse_new_bound(mol):
    return mol_parse_new(mol, rdmolfiles, np, el_mass)


def mol_write_yp_new_bound(file, elements, geo, bond_mat, adj_mat, atom_info=None):
    return mol_write_yp_new(file, elements, geo, bond_mat, adj_mat, atom_info, return_formals, np)


In [4]:
def header(title):
    print('\n' + '=' * 100)
    print(title)
    print('=' * 100)


def preview(value):
    if isinstance(value, np.ndarray):
        print('ndarray shape=', value.shape)
        print(value)
        return
    pprint(value)


def compare_block(name, old_call, new_call):
    header(name)
    print('OLD:')
    try:
        old_value = old_call()
        preview(old_value)
    except Exception as e:
        print(type(e).__name__, e)
        traceback.print_exc(limit=1)
        old_value = None

    print('\nNEW:')
    try:
        new_value = new_call()
        preview(new_value)
    except Exception as e:
        print(type(e).__name__, e)
        traceback.print_exc(limit=1)
        new_value = None

    return old_value, new_value


def summarize_smiles_result(result):
    if result is None:
        return None
    adj, bem, atom_info = result
    return {
        'adj_shape': adj.shape,
        'adj': adj,
        'bem_shape': bem.shape,
        'bem': bem,
        'atom_info_type': type(atom_info).__name__,
        'atom_info': atom_info,
    }


## 1. `smiles2adjmat()` old vs new

These cells compare parser outputs directly on unmapped, mapped, aromatic, and stereo-bearing inputs.


In [5]:
for case_name, smiles in SMILES_CASES.items():
    def old_call(smiles=smiles):
        return summarize_smiles_result(smiles2adjmat_old(smiles, verbose=True))

    def new_call(smiles=smiles):
        return summarize_smiles_result(smiles2adjmat_new(smiles, verbose=True))

    compare_block(f'smiles2adjmat comparison: {case_name} :: {smiles}', old_call, new_call)



smiles2adjmat comparison: haa_unmapped :: O=CCO
OLD:
{'adj': array([[0, 1, 0, 0, 0, 0, 0, 0],
       [1, 0, 1, 0, 1, 0, 0, 0],
       [0, 1, 0, 1, 0, 1, 1, 0],
       [0, 0, 1, 0, 0, 0, 0, 1],
       [0, 1, 0, 0, 0, 0, 0, 0],
       [0, 0, 1, 0, 0, 0, 0, 0],
       [0, 0, 1, 0, 0, 0, 0, 0],
       [0, 0, 0, 1, 0, 0, 0, 0]]),
 'adj_shape': (8, 8),
 'atom_info': [['O', 0, None, None, None, True],
               ['C', 0, None, None, None, True],
               ['C', 0, None, None, None, True],
               ['O', 0, None, None, None, True],
               ['H', 0, None, None, None, True],
               ['H', 0, None, None, None, True],
               ['H', 0, None, None, None, True],
               ['H', 0, None, None, None, True]],
 'atom_info_type': 'list',
 'bem': array([[4., 2., 0., 0., 0., 0., 0., 0.],
       [2., 0., 1., 0., 1., 0., 0., 0.],
       [0., 1., 0., 1., 0., 1., 1., 0.],
       [0., 0., 1., 4., 0., 0., 0., 1.],
       [0., 1., 0., 0., 0., 0., 0., 0.],
       [0., 0., 1

Traceback (most recent call last):
  File "/var/folders/qc/fg92hjk146s3g36x68b8n1lh0000gr/T/ipykernel_12621/1969604853.py", line 19, in compare_block
    old_value = old_call()
                ^^^^^^^^^^
yarp.yarpecule.graph.smiles.OctetError: Atom indices 3 has an octet violation.
Traceback (most recent call last):
  File "/var/folders/qc/fg92hjk146s3g36x68b8n1lh0000gr/T/ipykernel_12621/1969604853.py", line 28, in compare_block
    new_value = new_call()
                ^^^^^^^^^^
yarp.yarpecule.graph.smiles.OctetError: Atom indices 3 has an octet violation.


## 2. `mol_parse()` old vs new


In [6]:
for mol_file in [MOL_ETHANOL, MOL_ACETATE]:
    compare_block(
        f'mol_parse comparison: {mol_file.name}',
        lambda mol_file=mol_file: mol_parse_old(str(mol_file), rdmolfiles, np),
        lambda mol_file=mol_file: mol_parse_new(str(mol_file), rdmolfiles, np, el_mass),
    )



mol_parse comparison: ethanol.mol
OLD:
(['C', 'C', 'O'],
 array([[-0.9254,  0.0742,  0.0328],
       [ 0.5123, -0.4192, -0.0743],
       [ 1.3778,  0.4494,  0.6044]]),
 array([[0., 1., 0.],
       [1., 0., 1.],
       [0., 1., 0.]]),
 0)

NEW:
(['C', 'C', 'O'],
 array([[-0.9254,  0.0742,  0.0328],
       [ 0.5123, -0.4192, -0.0743],
       [ 1.3778,  0.4494,  0.6044]]),
 array([[0., 1., 0.],
       [1., 0., 1.],
       [0., 1., 0.]]),
 0,
 {0: {'aromatic_input': False,
      'atom_index': 0,
      'atom_map': None,
      'element': 'c',
      'formal_charge': 0,
      'mass': 12.011,
      'stereo': {'atom': None, 'bonds': {}}},
  1: {'aromatic_input': False,
      'atom_index': 1,
      'atom_map': None,
      'element': 'c',
      'formal_charge': 0,
      'mass': 12.011,
      'stereo': {'atom': None, 'bonds': {}}},
  2: {'aromatic_input': False,
      'atom_index': 2,
      'atom_map': None,
      'element': 'o',
      'formal_charge': 0,
      'mass': 15.9994,
      'stereo': {'a

## 3. `xyz_from_smiles()` old vs new


In [7]:
for mode in ('yarp', 'rdkit'):
    compare_block(
        f'xyz_from_smiles comparison: mode={mode}',
        lambda mode=mode: xyz_from_smiles_old(
            SMILES_CASES['haa_unmapped'], mode, smiles2adjmat_old, BondType, MolFromSmiles, rdchem, Atom,
            el_to_an, el_n_expand_octet, el_expand_octet, OctetError, AddHs, AllChem, np
        ),
        lambda mode=mode: xyz_from_smiles_new_bound(SMILES_CASES['haa_unmapped'], mode),
    )



xyz_from_smiles comparison: mode=yarp
OLD:
(['o', 'c', 'c', 'o', 'h', 'h', 'h', 'h'],
 array([[ 1.55474243,  0.73894876, -0.27185232],
       [ 0.93588664, -0.17231101,  0.25356394],
       [-0.52742591, -0.34098728, -0.01765354],
       [-0.8655664 ,  0.70051764, -0.90383715],
       [ 1.42291497, -0.8850771 ,  0.92983314],
       [-1.13172583, -0.25633354,  0.89061211],
       [-0.67654366, -1.33143462, -0.48260769],
       [-0.71228224,  1.54667715, -0.39805849]]),
 array([[0, 1, 0, 0, 0, 0, 0, 0],
       [1, 0, 1, 0, 1, 0, 0, 0],
       [0, 1, 0, 1, 0, 1, 1, 0],
       [0, 0, 1, 0, 0, 0, 0, 1],
       [0, 1, 0, 0, 0, 0, 0, 0],
       [0, 0, 1, 0, 0, 0, 0, 0],
       [0, 0, 1, 0, 0, 0, 0, 0],
       [0, 0, 0, 1, 0, 0, 0, 0]]),
 0)

NEW:
(['o', 'c', 'c', 'o', 'h', 'h', 'h', 'h'],
 array([[ 1.55474243,  0.73894876, -0.27185232],
       [ 0.93588664, -0.17231101,  0.25356394],
       [-0.52742591, -0.34098728, -0.01765354],
       [-0.8655664 ,  0.70051764, -0.90383715],
       [ 1.42

## 4. `_read_structure()` old vs new on SMILES, mapped SMILES, and XYZ


In [8]:
class DummyY:
    pass


def summarize_dummy(obj):
    return {
        'elements': getattr(obj, '_elements', None),
        'q': getattr(obj, '_q', None),
        'has_atom_info': getattr(obj, '_atom_info', None) is not None,
        'atom_info_type': type(getattr(obj, '_atom_info', None)).__name__,
        'atom_info': getattr(obj, '_atom_info', None),
    }


read_cases = [
    ('smiles_unmapped', SMILES_CASES['haa_unmapped']),
    ('smiles_mapped', SMILES_CASES['haa_mapped']),
    ('xyz_file', str(XYZ_HAA)),
]

for case_name, mol_input in read_cases:
    def old_call(mol_input=mol_input):
        obj = DummyY()
        yarpecule_read_structure_old(obj, mol_input, 'yarp', np, xyz_parse, table_generator, xyz_q_parse, mol_parse, xyz_from_smiles, el_mass)
        return summarize_dummy(obj)

    def new_call(mol_input=mol_input):
        obj = DummyY()
        yarpecule_read_structure_new(obj, mol_input, 'yarp', False, np, xyz_parse, table_generator, xyz_q_parse, mol_parse_new_bound, xyz_from_smiles_new_bound, el_mass)
        return summarize_dummy(obj)

    compare_block(f'_read_structure comparison: {case_name}', old_call, new_call)



_read_structure comparison: smiles_unmapped
OLD:
{'atom_info': None,
 'atom_info_type': 'NoneType',
 'elements': ['o', 'c', 'c', 'o', 'h', 'h', 'h', 'h'],
 'has_atom_info': False,
 'q': 0}

NEW:
TypeError yarpecule_read_structure_new() missing 1 required positional argument: 'el_mass'

_read_structure comparison: smiles_mapped
OLD:
{'atom_info': None,
 'atom_info_type': 'NoneType',
 'elements': ['c', 'c', 'o', 'o', 'h', 'h', 'h', 'h'],
 'has_atom_info': False,
 'q': 0}

NEW:
TypeError yarpecule_read_structure_new() missing 1 required positional argument: 'el_mass'

_read_structure comparison: xyz_file
OLD:
{'atom_info': None,
 'atom_info_type': 'NoneType',
 'elements': ['o', 'c', 'c', 'o', 'h', 'h', 'h', 'h'],
 'has_atom_info': False,
 'q': 0}

NEW:
TypeError yarpecule_read_structure_new() missing 1 required positional argument: 'el_mass'


Traceback (most recent call last):
  File "/var/folders/qc/fg92hjk146s3g36x68b8n1lh0000gr/T/ipykernel_12621/1969604853.py", line 28, in compare_block
    new_value = new_call()
                ^^^^^^^^^^
TypeError: yarpecule_read_structure_new() missing 1 required positional argument: 'el_mass'
Traceback (most recent call last):
  File "/var/folders/qc/fg92hjk146s3g36x68b8n1lh0000gr/T/ipykernel_12621/1969604853.py", line 28, in compare_block
    new_value = new_call()
                ^^^^^^^^^^
TypeError: yarpecule_read_structure_new() missing 1 required positional argument: 'el_mass'
Traceback (most recent call last):
  File "/var/folders/qc/fg92hjk146s3g36x68b8n1lh0000gr/T/ipykernel_12621/1969604853.py", line 28, in compare_block
    new_value = new_call()
                ^^^^^^^^^^
TypeError: yarpecule_read_structure_new() missing 1 required positional argument: 'el_mass'


## 5. `get_smiles()` old vs new on real yarpecules

The new path should preserve user maps and print verbose RDKit dumps.


In [9]:
def build_real_y(smiles_or_file, mode='yarp'):
    return yarpecule(smiles_or_file, mode=mode, canon=False)


real_cases = [
    ('real_unmapped_smiles', SMILES_CASES['haa_unmapped']),
    ('real_mapped_smiles', SMILES_CASES['haa_mapped']),
    ('real_xyz_input', str(XYZ_HAA)),
]

for case_name, mol_input in real_cases:
    def old_call(mol_input=mol_input):
        y = build_real_y(mol_input)
        yarpecule_get_smiles_old(y, os, Chem, mol_write_yp)
        return {'canon_smi': y.canon_smi, 'map_smi': y.map_smi, '_atom_info': y._atom_info}

    def new_call(mol_input=mol_input):
        y = build_real_y(mol_input)
        yarpecule_get_smiles_new(y, os, Chem, mol_write_yp_new_bound, verbose=True)
        return {'canon_smi': y.canon_smi, 'map_smi': y.map_smi, '_atom_info': y._atom_info}

    compare_block(f'get_smiles comparison: {case_name}', old_call, new_call)



get_smiles comparison: real_unmapped_smiles
OLD:
{'_atom_info': {},
 'canon_smi': 'O=CCO',
 'map_smi': '[O:0]=[C:1]([C:2]([O:3][H:7])([H:5])[H:6])[H:4]'}

NEW:
KeyError 0

get_smiles comparison: real_mapped_smiles
OLD:
{'_atom_info': {},
 'canon_smi': 'O=CCO',
 'map_smi': '[C:0]([C:1]([O:3][H:4])([H:6])[H:7])(=[O:2])[H:5]'}

NEW:
KeyError 0

get_smiles comparison: real_xyz_input
OLD:
{'_atom_info': {},
 'canon_smi': 'O=CCO',
 'map_smi': '[O:0]=[C:1]([C:2]([O:3][H:7])([H:5])[H:6])[H:4]'}

NEW:
KeyError 0


Traceback (most recent call last):
  File "/var/folders/qc/fg92hjk146s3g36x68b8n1lh0000gr/T/ipykernel_12621/1969604853.py", line 28, in compare_block
    new_value = new_call()
                ^^^^^^^^^^
KeyError: 0
Traceback (most recent call last):
  File "/var/folders/qc/fg92hjk146s3g36x68b8n1lh0000gr/T/ipykernel_12621/1969604853.py", line 28, in compare_block
    new_value = new_call()
                ^^^^^^^^^^
KeyError: 0
Traceback (most recent call last):
  File "/var/folders/qc/fg92hjk146s3g36x68b8n1lh0000gr/T/ipykernel_12621/1969604853.py", line 28, in compare_block
    new_value = new_call()
                ^^^^^^^^^^
KeyError: 0


## 6. `export_geometry()` old vs new

This writes development files under `dev/generated/` so the outputs can be inspected directly.


In [10]:
y_old = build_real_y(SMILES_CASES['haa_mapped'])
y_new = build_real_y(SMILES_CASES['haa_mapped'])
yarpecule_get_smiles_old(y_old, os, Chem, mol_write_yp)
yarpecule_get_smiles_new(y_new, os, Chem, mol_write_yp_new_bound, verbose=True)

old_xyz = GENERATED_DIR / 'old_export.xyz'
old_mol = GENERATED_DIR / 'old_export.mol'
new_xyz = GENERATED_DIR / 'new_export.xyz'
new_mol = GENERATED_DIR / 'new_export.mol'

yarpecule_export_geometry_old(y_old, str(old_xyz), 'xyz', xyz_write, mol_write_yp)
yarpecule_export_geometry_old(y_old, str(old_mol), 'mol', xyz_write, mol_write_yp)
yarpecule_export_geometry_new(y_new, str(new_xyz), 'xyz', xyz_write_new, mol_write_yp_new_bound)
yarpecule_export_geometry_new(y_new, str(new_mol), 'mol', xyz_write_new, mol_write_yp_new_bound)

for path in [old_xyz, old_mol, new_xyz, new_mol]:
    header(f'file contents: {path.name}')
    print(path.read_text())


KeyError: 0

## 7. Known caveats while using this notebook

- Some draft functions in `dev/edit_plan.py` are still review artifacts, not guaranteed runnable production replacements.
- The old/new comparison is meant to show drift in inputs, outputs, and metadata shape before implementation.
- The most important comparison points are:
  - whether `_atom_info` exists
  - whether user atom maps reach `map_smi`
  - whether aromatic and stereo-bearing inputs fail or survive
  - whether exported XYZ/MOL content changes as expected
